# 微调模型回答质量验证

本 notebook 独立于训练流程，专门用来分析微调后 GPT-2 模型的回答质量。

**目的：** 区分「模型生成质量差」还是「评估方法有问题」

分析维度：
1. 人工抽样检查：直观感受回答质量
2. 简单自动指标：文本重叠度、回答长度等
3. LLM-as-Judge：用智谱 GLM-4-Flash API 评分（透明化每条得分）

In [ ]:
import json
import re
import requests
from tqdm import tqdm

with open("instruction-data-with-response.json", "r") as f:
    test_data = json.load(f)

print(f"共 {len(test_data)} 条测试数据")
print(f"字段: {list(test_data[0].keys())}")

## 1. 人工抽样检查

随机抽取 15 条数据，逐条对比「标准答案」和「模型回答」。

In [ ]:
import random
random.seed(42)
samples = random.sample(test_data, 15)

for i, entry in enumerate(samples):
    print(f"━━━ 样本 {i+1} ━━━")
    print(f"📋 指令: {entry['instruction']}")
    if entry['input']:
        print(f"📥 输入: {entry['input']}")
    print(f"✅ 标准: {entry['output']}")
    print(f"🤖 模型: {entry['model_response']}")
    print()

## 1.5 对比测试：原始 GPT-2（未微调） vs 微调后

加载原始 GPT-2 权重，用相同的指令生成回答，验证微调是否真的带来了改进。

In [ ]:
import torch
import tiktoken
import os
from transformers import GPT2LMHeadModel

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")

# 加载原始 GPT-2（未微调）
raw_model = GPT2LMHeadModel.from_pretrained("gpt2", cache_dir=os.path.join("..", "models", "gpt2"))
raw_model.eval().to(device)
print(f"原始 GPT-2 加载完毕 ✓ | 设备: {device}")

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

def generate_response(model, prompt, max_new_tokens=256):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    eos_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            do_sample=False, eos_token_id=eos_id,
            pad_token_id=eos_id
        )
    full_text = tokenizer.decode(output[0].tolist())
    return full_text[len(prompt):].replace("### Response:", "").strip()

CACHE_FILE = "raw-gpt2-responses.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        raw_responses = json.load(f)
    for i, entry in enumerate(test_data):
        test_data[i]["raw_gpt2_response"] = raw_responses[i]["raw_gpt2_response"]
    del raw_model
    print(f"从缓存加载原始 GPT-2 回答 ✓ ({CACHE_FILE})")
else:
    print("正在用原始 GPT-2 生成回答...")
    for i, entry in tqdm(enumerate(test_data), total=len(test_data), desc="原始模型生成"):
        prompt = format_input(entry) + "\n\n### Response:\n"
        test_data[i]["raw_gpt2_response"] = generate_response(raw_model, prompt)
    del raw_model
    with open(CACHE_FILE, "w") as f:
        json.dump([{"instruction": e["instruction"], "raw_gpt2_response": e["raw_gpt2_response"]} for e in test_data], f, indent=2)
    print(f"原始 GPT-2 回答已缓存到 {CACHE_FILE} ✓")

torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# 抽样对比：原始 vs 微调
random.seed(42)
compare_samples = random.sample(test_data, 10)

for i, entry in enumerate(compare_samples):
    print(f"━━━ 样本 {i+1} ━━━")
    print(f"📋 指令: {entry['instruction']}")
    if entry['input']:
        print(f"📥 输入: {entry['input']}")
    print(f"✅ 标准: {entry['output']}")
    print(f"🔴 原始: {entry['raw_gpt2_response'][:120]}")
    print(f"🟢 微调: {entry['model_response'][:120]}")
    print()

In [ ]:
# 量化对比：原始 vs 微调
stop_words = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in", "for", "on", "with", "that", "this", "it", "and", "or", "be"}

def calc_metrics(test_data, response_key):
    exact, keyword, empty = 0, 0, 0
    for entry in test_data:
        resp = entry[response_key].strip().lower()
        expected = entry["output"].strip().lower()
        if not resp:
            empty += 1
            continue
        if resp == expected:
            exact += 1
        keywords = [w for w in expected.split() if w not in stop_words and len(w) > 2]
        if keywords and sum(1 for kw in keywords if kw in resp) / len(keywords) >= 0.5:
            keyword += 1
    return exact, keyword, empty

ft_exact, ft_kw, ft_empty = calc_metrics(test_data, "model_response")
raw_exact, raw_kw, raw_empty = calc_metrics(test_data, "raw_gpt2_response")

n = len(test_data)
print(f"{'指标':<20} {'原始 GPT-2':>12} {'微调后':>12}")
print("─" * 48)
print(f"{'完全匹配':<20} {raw_exact:>8}/{n}   {ft_exact:>8}/{n}")
print(f"{'关键词匹配(≥50%)':<20} {raw_kw:>8}/{n}   {ft_kw:>8}/{n}")
print(f"{'空回答':<20} {raw_empty:>8}/{n}   {ft_empty:>8}/{n}")

# 检查原始模型是否在「遵循指令」（而非续写/重复指令）
follows_format = 0
repeats_instruction = 0
for entry in test_data:
    resp = entry["raw_gpt2_response"].strip()
    if "### Instruction" in resp or "### Input" in resp:
        repeats_instruction += 1
    elif len(resp) > 5 and "### " not in resp:
        follows_format += 1

ft_follows = sum(1 for e in test_data if len(e["model_response"].strip()) > 5 and "### " not in e["model_response"])

print(f"\n{'遵循指令格式':<20} {follows_format:>8}/{n}   {ft_follows:>8}/{n}")
print(f"{'重复/续写指令':<20} {repeats_instruction:>8}/{n}   {'—':>8}")
print(f"\n💡 原始 GPT-2 大概率会续写/重复指令模板，而非真正回答问题。")
print(f"   微调的核心价值：让模型学会「看到指令就生成回答」的格式。")

## 2. 简单自动指标

不依赖外部 LLM，用文本匹配方式快速评估：
- **完全匹配率**：模型回答 == 标准答案
- **包含率**：标准答案的关键词出现在模型回答中
- **回答长度分布**：模型是否在合理长度范围内

In [ ]:
exact_match = 0
keyword_match = 0
empty_response = 0
response_lengths = []
expected_lengths = []

for entry in test_data:
    resp = entry["model_response"].strip().lower()
    expected = entry["output"].strip().lower()

    if not resp:
        empty_response += 1
        continue

    response_lengths.append(len(resp.split()))
    expected_lengths.append(len(expected.split()))

    if resp == expected:
        exact_match += 1

    # 提取标准答案中的关键词（去除停用词后取词频最高的）
    stop_words = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in", "for", "on", "with", "that", "this", "it", "and", "or", "be"}
    keywords = [w for w in expected.split() if w not in stop_words and len(w) > 2]
    if keywords:
        matched = sum(1 for kw in keywords if kw in resp)
        if matched / len(keywords) >= 0.5:
            keyword_match += 1

n = len(test_data)
print(f"完全匹配:  {exact_match}/{n} ({exact_match/n*100:.1f}%)")
print(f"关键词匹配(≥50%): {keyword_match}/{n} ({keyword_match/n*100:.1f}%)")
print(f"空回答:    {empty_response}/{n}")
print(f"\n回答平均词数: {sum(response_lengths)/len(response_lengths):.1f} (标准答案: {sum(expected_lengths)/len(expected_lengths):.1f})")

## 3. 分类统计：哪些答对了，哪些答错了

In [ ]:
good, partial, bad = [], [], []

stop_words = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in", "for", "on", "with", "that", "this", "it", "and", "or", "be"}

for entry in test_data:
    resp = entry["model_response"].strip().lower()
    expected = entry["output"].strip().lower()
    keywords = [w for w in expected.split() if w not in stop_words and len(w) > 2]

    if resp == expected:
        good.append(entry)
    elif keywords and sum(1 for kw in keywords if kw in resp) / len(keywords) >= 0.5:
        partial.append(entry)
    else:
        bad.append(entry)

print(f"✅ 优秀（完全匹配）: {len(good)} 条 ({len(good)/len(test_data)*100:.1f}%)")
print(f"🟡 部分正确（关键词≥50%）: {len(partial)} 条 ({len(partial)/len(test_data)*100:.1f}%)")
print(f"❌ 较差（关键词<50%）: {len(bad)} 条 ({len(bad)/len(test_data)*100:.1f}%)")

print(f"\n--- ❌ 随机 5 条较差回答 ---")
for entry in random.sample(bad, min(5, len(bad))):
    print(f"  指令: {entry['instruction'][:60]}")
    print(f"  标准: {entry['output'][:60]}")
    print(f"  模型: {entry['model_response'][:60]}")
    print()

## 4. LLM-as-Judge 评分（透明化）

使用智谱 AI **GLM-4-Flash**（免费）逐条评分，打印每条的分数，方便判断评估是否合理。

> 需要安装 `zhipuai` 并配置 API Key。

In [ ]:
from zhipuai import ZhipuAI

ZHIPU_API_KEY = "d63eddac939145a1a2564bfb51a58ac9.OM0UPDTi3lBFNMcu"
EVAL_MODEL = "glm-4-flash"

client = ZhipuAI(api_key=ZHIPU_API_KEY)

def query_model(prompt):
    """调用 GLM-4-Flash API 进行评分"""
    response = client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip()

# 先用 5 条测试评估是否正常工作
print("=== 前 5 条评分测试 (GLM-4-Flash) ===")
for entry in test_data[:5]:
    prompt = (f"Given the input `{entry['instruction']}` and correct output `{entry['output']}`, "
              f"score the model response `{entry['model_response']}` on a scale from 0 to 100, "
              f"where 100 is the best score. Respond with the integer number only.")
    raw = query_model(prompt)
    print(f"  指令: {entry['instruction'][:50]}")
    print(f"  标准: {entry['output'][:50]}")
    print(f"  模型: {entry['model_response'][:50]}")
    print(f"  原始返回: [{raw}] → 解析: ", end="")
    try:
        score = int(re.search(r'\d+', raw).group())
        print(f"{score} 分")
    except (ValueError, AttributeError):
        print(f"解析失败")
    print()

In [ ]:
# 全量评估
scores = []
failed = 0

for entry in tqdm(test_data, desc="GLM-4-Flash 评估中"):
    prompt = (f"Given the input `{entry['instruction']}` and correct output `{entry['output']}`, "
              f"score the model response `{entry['model_response']}` on a scale from 0 to 100, "
              f"where 100 is the best score. Respond with the integer number only.")
    raw_response = query_model(prompt)
    try:
        score = int(re.search(r'\d+', raw_response).group())
        scores.append({"instruction": entry["instruction"], "score": score})
    except (ValueError, AttributeError):
        failed += 1
        continue

print(f"\n有效评分: {len(scores)}/{len(test_data)} | 解析失败: {failed}")
if scores:
    all_scores = [s["score"] for s in scores]
    print(f"平均分: {sum(all_scores)/len(all_scores):.2f} / 100")
    print(f"最高分: {max(all_scores)} | 最低分: {min(all_scores)} | 中位数: {sorted(all_scores)[len(all_scores)//2]}")

    print(f"\n--- 分数分布 ---")
    brackets = [(0,20), (20,40), (40,60), (60,80), (80,101)]
    for lo, hi in brackets:
        count = sum(1 for s in all_scores if lo <= s < hi)
        bar = "█" * count
        print(f"  {lo:>3}-{min(hi,100):<3}: {bar} ({count})")

## 5. 指令遵循度测试（忽略答案正确性）

指令微调的核心目标不是让 124M 参数的模型记住事实，而是让它学会：
1. **不重复/续写指令** — 看到 `### Response:` 后生成回答，而非重复模板
2. **回答与指令相关** — 提到了指令中的主题/关键词
3. **回答类型正确** — 问"什么是"就给陈述句，要求"改写"就给改写后的句子
4. **回答格式合理** — 长度适中、语句完整、正常结束

下面分别测试原始 GPT-2 和微调后模型在这些维度上的表现。

In [ ]:
def eval_instruction_following(test_data, response_key):
    """评估指令遵循度（不关心答案正确性）"""
    results = {
        "not_repeating": 0,     # 没有重复/续写指令模板
        "topic_relevant": 0,    # 回答与指令主题相关
        "type_correct": 0,      # 回答类型与指令要求匹配
        "proper_length": 0,     # 长度合理（3-100词）
        "complete_sentence": 0, # 是完整句子（非截断/乱码）
        "uses_eos": 0,          # 正常结束（非被截断到 max_tokens）
    }
    details = []

    for entry in test_data:
        resp = entry[response_key].strip()
        resp_lower = resp.lower()
        instruction = entry["instruction"].lower()
        input_text = entry.get("input", "").lower()

        item = {"instruction": entry["instruction"], "response": resp[:80]}

        # 1. 不重复指令：回答中不包含指令模板标记
        not_repeating = "### instruction" not in resp_lower and "### input" not in resp_lower
        if not_repeating:
            # 进一步检查：回答不是直接复制指令文本
            not_repeating = resp_lower.strip() != instruction.strip()
        results["not_repeating"] += int(not_repeating)
        item["not_repeating"] = not_repeating

        # 2. 主题相关：指令中的实体/名词出现在回答中
        stop_words = {"the","a","an","is","are","was","were","of","to","in","for",
                      "on","with","that","this","it","and","or","be","what","how",
                      "which","where","when","who","does","do","did","can","could",
                      "would","should","will","following","sentence","word","using",
                      "write","create","generate","identify","describe","explain",
                      "provide","give","name","list","rewrite","convert","correct"}
        topic_words = [w.strip(".,;:!?'\"") for w in instruction.split()
                       if w.strip(".,;:!?'\"").lower() not in stop_words and len(w) > 2]
        if input_text:
            topic_words += [w.strip(".,;:!?'\"") for w in input_text.split()
                           if w.strip(".,;:!?'\"").lower() not in stop_words and len(w) > 2]
        if topic_words:
            topic_hit = sum(1 for tw in topic_words if tw in resp_lower) / len(topic_words)
            topic_relevant = topic_hit >= 0.3
        else:
            topic_relevant = len(resp) > 5
        results["topic_relevant"] += int(topic_relevant)
        item["topic_relevant"] = topic_relevant

        # 3. 回答类型匹配
        type_correct = True
        if any(kw in instruction for kw in ["opposite", "synonym", "antonym"]):
            type_correct = len(resp.split()) <= 15
        elif any(kw in instruction for kw in ["rewrite", "convert", "correct"]):
            type_correct = not resp_lower.startswith("the ") or "should" not in resp_lower
        elif "sentence" in instruction and "generate" in instruction:
            type_correct = 5 <= len(resp.split()) <= 30
        results["type_correct"] += int(type_correct)
        item["type_correct"] = type_correct

        # 4. 长度合理（3-100词，既不是空的也不是无限续写）
        word_count = len(resp.split())
        proper_length = 1 <= word_count <= 100
        results["proper_length"] += int(proper_length)
        item["proper_length"] = proper_length

        # 5. 完整句子（以标点结束，或者是短答案）
        complete = (resp.endswith(('.', '!', '?', '"', "'", ')'))
                    or word_count <= 5
                    or resp.endswith(('。', '！', '？')))
        results["complete_sentence"] += int(complete)
        item["complete_sentence"] = complete

        # 6. 正常结束（没有在句子中间截断——检查是否到了 max_tokens 附近还没结束）
        uses_eos = word_count < 200
        results["uses_eos"] += int(uses_eos)
        item["uses_eos"] = uses_eos

        details.append(item)

    return results, details

# 分别评估
ft_results, ft_details = eval_instruction_following(test_data, "model_response")
raw_results, raw_details = eval_instruction_following(test_data, "raw_gpt2_response")

n = len(test_data)
labels = {
    "not_repeating": "① 不重复指令",
    "topic_relevant": "② 主题相关",
    "type_correct": "③ 回答类型正确",
    "proper_length": "④ 长度合理(1-100词)",
    "complete_sentence": "⑤ 完整句子",
    "uses_eos": "⑥ 正常结束",
}

print(f"{'指令遵循度指标':<22} {'原始 GPT-2':>14} {'微调后':>14} {'提升':>10}")
print("─" * 64)
for key, label in labels.items():
    raw_pct = raw_results[key] / n * 100
    ft_pct = ft_results[key] / n * 100
    delta = ft_pct - raw_pct
    arrow = "↑" if delta > 0 else "↓" if delta < 0 else "—"
    print(f"{label:<22} {raw_results[key]:>6}/{n} ({raw_pct:4.1f}%) {ft_results[key]:>6}/{n} ({ft_pct:4.1f}%) {arrow}{abs(delta):5.1f}%")

# 综合得分
raw_total = sum(raw_results.values()) / (n * len(raw_results)) * 100
ft_total = sum(ft_results.values()) / (n * len(ft_results)) * 100
print("─" * 64)
print(f"{'综合遵循度':<22} {raw_total:>14.1f}% {ft_total:>14.1f}% {'↑' if ft_total > raw_total else '↓'}{abs(ft_total-raw_total):5.1f}%")

In [ ]:
# 展示微调后仍然失败的案例（哪些维度没通过）
print("=== 微调后仍未通过的案例 ===\n")
fail_count = 0
for d in ft_details:
    failed_dims = [k for k in ["not_repeating", "topic_relevant", "type_correct", "proper_length", "complete_sentence"]
                   if not d[k]]
    if failed_dims:
        fail_count += 1
        if fail_count <= 8:
            dim_names = {"not_repeating":"重复指令", "topic_relevant":"主题无关",
                         "type_correct":"类型错误", "proper_length":"长度异常", "complete_sentence":"句子不完整"}
            print(f"  指令: {d['instruction'][:55]}")
            print(f"  回答: {d['response'][:55]}")
            print(f"  问题: {', '.join(dim_names[k] for k in failed_dims)}")
            print()

print(f"共 {fail_count}/{n} 条在至少一个维度上未通过")
print(f"完全通过所有检查: {n - fail_count}/{n} ({(n-fail_count)/n*100:.1f}%)")

## 5. 结论

对比上面的分析结果：

- 如果「简单指标」和「LLM 评分」都低 → **模型生成质量确实有限**（GPT-2 124M 参数量小，知识储备不足）
- 如果「简单指标」还行但「LLM 评分」很低 → **评估方法可能偏严或有问题**
- 如果「LLM 评分」的原始返回经常不是整数 → **评估模型理解力不足，需要换更大模型**

> 💡 GPT-2 Small (124M) 的指令微调效果有限是正常的——它的参数量太小，知识容量不足以回答事实性问题。
> 指令微调的价值在于让模型学会了「指令→回答」的格式，而非记住具体知识。